# Sample Size and Columns
### Why every 40th row , the 0.025% cut and the dropped columns.
### Plain logistic regression from `self_module/logistic.py` , 500 epochs at max_lr , scored on the same validation rows every time (every 40th row of valid). Test is never read here.

In [1]:
import os
import sys
import time
import zipfile
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import log_loss , roc_auc_score

sys.path.append('../..')
from self_module import LogisticRegression

### Reading every train row once , each sample size keeps its own rows.

In [2]:
# chunksize is a multiple of every step , so each pattern carries across chunks
steps = [128 , 64 , 45 , 40 , 32 , 20]
drop = ['id' , 'device_id' , 'device_ip']
if not os.path.exists('../../data/ctr/sweep_20.parquet'):
    parts = {k : [] for k in steps}
    with zipfile.ZipFile('../../data/Dataset1.zip') as z:
        for chunk in pd.read_csv(z.open('train.csv') , dtype=str , chunksize=1_440_000 , usecols=lambda c : c not in drop):
            for k in steps:
                parts[k].append(chunk.iloc[::k])
    for k in steps:
        s = pd.concat(parts[k] , ignore_index=True)
        s['click'] = s['click'].astype(int)
        s.to_parquet(f'../../data/ctr/sweep_{k}.parquet' , index=False)

def load(path):
    d = pd.read_parquet(path)
    d['hour_of_day'] = d['hour'].str[6:]
    d['weekday'] = pd.to_datetime(d['hour'] , format='%y%m%d%H').dt.day_name()
    return d

valid = load('../../data/ctr/valid_sample.parquet')
y_val = torch.tensor(valid['click'].values , dtype=torch.float32).reshape(-1 , 1)

### Same steps as 02_data_prep (rare values into other , first value dropped , standardised with train) , done on numpy so the biggest samples fit in RAM.

In [3]:
def encode(train , cut , feats):
    plans , width = {} , 0
    for f in feats:
        share = train[f].value_counts(normalize=True)
        keep = share[share >= cut].index
        values = sorted(train[f].where(train[f].isin(keep) , 'other').unique())
        plans[f] = (keep , values , width)
        width += len(values) - 1

    def one_hot(d):
        X = np.zeros((len(d) , width + 1) , dtype='float32')
        for f , (keep , values , start) in plans.items():
            c = pd.Categorical(d[f].where(d[f].isin(keep) , 'other') , categories=values).codes.astype('int64')
            rows = np.nonzero(c > 0)[0]
            X[rows , start + c[rows] - 1] = 1
        return X

    X_train , X_val = one_hot(train) , one_hot(valid)
    # a 1 / 0 column has std = sqrt(mean (1 - mean)) , so no float64 copy of the whole matrix
    mean = X_train.mean(0 , dtype='float64')
    std = np.sqrt(mean * (1 - mean))
    std[std == 0] = 1
    for X in [X_train , X_val]:
        X -= mean.astype('float32')
        X /= std.astype('float32')
        X[: , -1] = 1
    blocks = {f : (start , start + len(values) - 1) for f , (keep , values , start) in plans.items()}
    return X_train , X_val , blocks

### The notebooks keep train , valid and test on the GPU , so that is what "fits" tries. Anything that doesn't fit is trained on the CPU , same maths , only slower.

In [4]:
def fits(n_train , n_val , d):
    try:
        X = torch.zeros((n_train , d) , dtype=torch.float32 , device='cuda')
        y = torch.zeros((n_train , 1) , dtype=torch.float32 , device='cuda')
        others = [torch.zeros((n_val , d) , dtype=torch.float32 , device='cuda') for _ in range(2)]
        m = LogisticRegression(lr=0.001 , epochs=1)
        m.max_lr(X)
        m.train(X , y)
        return True
    except torch.OutOfMemoryError:
        return False

def run(X_train , y_train , X_val):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    on_gpu = fits(len(X_train) , len(X_val) , X_train.shape[1])
    peak = torch.cuda.max_memory_allocated() / 1e9
    torch.cuda.empty_cache()
    device = 'cuda' if on_gpu else 'cpu'
    start = time.time()
    X = torch.from_numpy(X_train).to(device)
    y = torch.tensor(y_train , dtype=torch.float32 , device=device).reshape(-1 , 1)
    model = LogisticRegression(lr=LogisticRegression().max_lr(X) , epochs=500)
    model.train(X , y)
    p = model.forward(torch.from_numpy(X_val).to(device)).cpu().numpy().ravel()
    y_v = y_val.numpy().ravel()
    del X , y , model
    torch.cuda.empty_cache()
    return {'columns' : X_train.shape[1] - 1 , 'GPU peak GB' : round(peak , 2) if on_gpu else 'does not fit' ,
            'log loss' : log_loss(y_v , p) , 'roc_auc' : roc_auc_score(y_v , p) , 'minutes' : round((time.time() - start) / 60 , 1)}

## Sample size and cut
### All 18 columns from 02_data_prep before the last three drops (C18 , C19 , C21) , then the final setting.

In [5]:
all_18 = ['banner_pos' , 'site_id' , 'site_domain' , 'site_category' , 'app_id' , 'app_category' , 'device_model' , 'device_type' ,
          'device_conn_type' , 'C15' , 'C16' , 'C17' , 'C18' , 'C19' , 'C20' , 'C21' , 'hour_of_day' , 'weekday']
final = [f for f in all_18 if f not in ['C18' , 'C19' , 'C21']]

settings = [(128 , 0.001 , all_18) , (64 , 0.001 , all_18) , (32 , 0.001 , all_18) , (20 , 0.001 , all_18) ,
            (45 , 0.00025 , all_18) , (40 , 0.00025 , all_18) , (32 , 0.00025 , all_18) , (20 , 0.00025 , all_18) , (40 , 0.00025 , final)]
rows = []
for k , cut , feats in settings:
    train = load(f'../../data/ctr/sweep_{k}.parquet')
    X_train , X_val , _ = encode(train , cut , feats)
    r = run(X_train , train['click'].values , X_val)
    rows.append({'sample' : f'1 in {k}' , 'cut %' : cut * 100 , 'feature columns' : len(feats) , 'train rows' : len(train) , **r})
    del X_train , X_val
sample_sizes = pd.DataFrame(rows).round({'log loss' : 5 , 'roc_auc' : 5})
sample_sizes.to_csv('../../data/ctr/sweep_sample_sizes.csv' , index=False)
sample_sizes

,sample,cut %,feature columns,train rows,columns,GPU peak GB,log loss,roc_auc,minutes
0,1 in 128,0.100,18,252682,707,1.31,0.40671,0.72518,0.1
1,1 in 64,0.100,18,505363,705,2.02,0.40599,0.72641,0.2
2,1 in 32,0.100,18,1010725,703,3.45,0.40571,0.72705,0.3
3,1 in 20,0.100,18,1617159,703,5.17,0.40575,0.72706,0.5
4,1 in 45,0.025,18,718738,1475,5.5,0.40199,0.73443,0.5
5,1 in 40,0.025,18,808580,1464,does not fit,0.40176,0.73472,2.3
6,1 in 32,0.025,18,1010725,1468,does not fit,0.40146,0.73563,2.8
7,1 in 20,0.025,18,1617159,1468,does not fit,0.40131,0.73569,4.3
8,1 in 40,0.025,15,808580,1356,5.55,0.40190,0.73454,0.5


### More rows alone barely help : at the 0.1% cut , going from 1 in 128 to 1 in 32 gains 0.0010 , and 1 in 20 gains nothing more.
### The cut helps far more : at 1 in 32 , going from 0.1% to 0.025% lowers log loss by 0.0043 and raises ROC-AUC from 0.727 to 0.736.
### 0.025% needs about twice the columns , so the biggest sample that fits with all 18 is 1 in 45. Dropping C18 , C19 , C21 makes room for 1 in 40 , the final setting.
### It gives up 0.0006 against 1 in 20 at 0.025% (0.40190 vs 0.40131) , which doesn't fit.
### Every row here is scored on the same 101,073 validation rows , and the GPU peak counts valid and test at that size.
### All 32 million rows aren't in the table : one hot they need about 175 GB. That comes later , with batching.

### Why the cut helps : share of rows that end up in 'other' (1 in 40).

In [6]:
train = load('../../data/ctr/sweep_40.parquet')
other = {}
for f in ['device_model' , 'site_id' , 'site_domain' , 'app_id' , 'C17']:
    share = train[f].value_counts(normalize=True)
    other[f] = {'cut 0.1%' : share[share < 0.001].sum() , 'cut 0.025%' : share[share < 0.00025].sum()}
pd.DataFrame(other).T.round(3)

,cut 0.1%,cut 0.025%
device_model,0.296,0.131
site_id,0.116,0.058
site_domain,0.088,0.047
app_id,0.077,0.045
C17,0.068,0.015


### At 0.1% , 30% of device_model rows land in other , at 0.025% only 13%. That detail is what the lower cut gets back.

## Columns
### Every column against the final setting (1 in 40 , 0.025% cut). Cost = how much log loss rises without that column , so close to 0 means it can go.

In [7]:
extras = ['C1' , 'C14' , 'C18' , 'C19' , 'C21' , 'app_domain']
X_all , X_val_all , blocks = encode(train , 0.00025 , final + extras)
y_train = train['click'].values

def score(feats):
    mask = np.zeros(X_all.shape[1] , dtype=bool)
    mask[-1] = True
    for f in feats:
        mask[blocks[f][0] : blocks[f][1]] = True
    return run(X_all[: , mask] , y_train , X_val_all[: , mask])

base = score(final)
rows = []
for f in extras:
    r = score(final + [f])
    rows.append({'column' : f , 'in the final model' : 'no' , 'columns it takes' : r['columns'] - base['columns'] , 'cost' : base['log loss'] - r['log loss'] , 'GPU peak GB' : r['GPU peak GB']})
for f in final:
    r = score([g for g in final if g != f])
    rows.append({'column' : f , 'in the final model' : 'yes' , 'columns it takes' : base['columns'] - r['columns'] , 'cost' : r['log loss'] - base['log loss'] , 'GPU peak GB' : r['GPU peak GB']})
for name , feats in [('C18 , C19 , C21 together' , final + ['C18' , 'C19' , 'C21']) , ('C20 instead of C21' , [g for g in final if g != 'C20'] + ['C21'])]:
    r = score(feats)
    rows.append({'column' : name , 'in the final model' : 'no' , 'columns it takes' : r['columns'] - base['columns'] , 'cost' : base['log loss'] - r['log loss'] , 'GPU peak GB' : r['GPU peak GB']})
columns = pd.DataFrame(rows).sort_values('cost').round({'cost' : 5})
columns.to_csv('../../data/ctr/sweep_columns.csv' , index=False)
print('final model :' , base)
columns

final model : {'columns': 1356, 'GPU peak GB': 5.55, 'log loss': 0.40189517674753383, 'roc_auc': 0.7345353733159903, 'minutes': 0.5}


,column,in the final model,columns it takes,cost,GPU peak GB
22,C20 instead of C21,no,-43,-0.00004,5.37
15,C15,yes,4,-0.00001,5.53
20,weekday,yes,6,-0.00000,5.52
14,device_conn_type,yes,3,0.00000,5.53
0,C1,no,5,0.00001,5.57
2,C18,no,3,0.00001,5.56
13,device_type,yes,3,0.00002,5.53
5,app_domain,no,24,0.00006,5.64
4,C21,no,50,0.00007,5.75
6,banner_pos,yes,4,0.00009,5.53


### Final model : 1,356 columns , log loss 0.40190 , ROC-AUC 0.7345.
### Left out : C1 saves 5 columns for 0.00001 , app_domain 24 for 0.00006 , C18 , C19 , C21 save 108 for 0.00013 together. C14 saves 421 for 0.00094 , and the model doesn't fit with it.
### Kept because they cost the most to lose : app_id (0.00374) , C17 (0.00343) , device_model (0.00179) , site_id (0.00101).
### C15 , weekday , device_conn_type and device_type cost almost nothing too , but together they only save 16 columns , so they stay.
### Dropping C20 instead of C21 saves 43 more columns but log loss is 0.00004 higher , and C21 is the one C17 fixes.
### Gaps under about 0.0001 are too small to read much into on 101,073 rows.